# Baseline B3

Author: Mathias Marryat Husted

This notebook provides a reproducible way to run the baseline B3 runs. Checks are performed at each step to ensure data consistency.

Baseline B3 focuses on direct API matching.

In [1]:
import requests
print(requests.get("https://rest.genenames.org/info", headers={"Accept": "application/json"}).json()["lastModified"])

2026-10-06T12:16:24.016Z


API access date: 2026-10-07

HGNC last modified on 2026-10-06

In [2]:
import pandas as pd
import json
import urllib.request
import urllib.error
from urllib.parse import quote
import time

# Reload the benchmark files

BENCHMARK_PATH = "../../data/data set/benchmark.csv"
CONTROL_PATH = "../../data/data set/control.csv"
FIELD_ORDER = ["symbol", "prev_symbol", "alias_symbol", "name", "prev_name", "alias_name"] # Ruleset fallback order

benchmark = pd.read_csv(BENCHMARK_PATH, dtype=str, keep_default_na=False)
control = pd.read_csv(CONTROL_PATH, dtype=str, keep_default_na=False)


In [6]:
# Adapted functions from the resolution_auditor

def fetch_json(url, headers=None, retries=3):
    for attempt in range(retries):
        time.sleep(0.2)
        req = urllib.request.Request(url, headers=headers or {})
        try:
            with urllib.request.urlopen(req) as response:
                return json.load(response)
        except urllib.error.HTTPError as e:
            if e.code in (429, 500, 502, 503, 504) and attempt < retries - 1:
                time.sleep(4)
                continue
            if e.code == 404:
                try:
                    return json.load(e)
                except json.JSONDecodeError:
                    return {} # This happened once in testing and lead to the run being invalid
            raise Exception("Error")
        except urllib.error.URLError:
            if attempt < retries - 1:
                time.sleep(4)
                continue # This error happened to a network dropout in testing. Trying to catch it this way
            raise Exception("Error")


def hgnc_query(path):
    result = fetch_json(f"https://rest.genenames.org/{path}", {"Accept": "application/json"})
    if result == {}:
        return None
    if result is None or "response" not in result:
        raise ValueError("Unexpected response type")
    return result["response"]["docs"]

def get_gene_ids(name):
    result = []
    matched_field = ""
    for field in FIELD_ORDER:
        docs = hgnc_query(f"fetch/{field}/{quote(name, safe='')}")
        if docs is None:
            return [], "error_query"
        for doc in docs:
            values = doc.get(field)
            if isinstance(values, str):
                values = [values]
            for value in values:
                if doc.get("status") == "Approved" and value.lower() == name.lower():
                    result.append(doc["hgnc_id"])
        if result:
            matched_field = field
            break
    return result, matched_field


# Resolve a chemical name to every CID the public PubChem name search returns
def get_chemical_ids(name):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{quote(name, safe='')}/cids/JSON"
    result = fetch_json(url)
    if "IdentifierList" not in result:
        return []
    return result["IdentifierList"]["CID"]


def resolve_chemical(name):
    id_hits = get_chemical_ids(name)
    distinct_ids = []
    for hit in id_hits:
        if hit not in distinct_ids:
            distinct_ids.append(hit)

    return {
        "n_candidates": len(distinct_ids) if distinct_ids is not None else 0,
        "returned_id": str(distinct_ids[0]) if len(distinct_ids) == 1 else "",
        "candidate_ids": ";".join(map(str, distinct_ids)),
    }

def resolve_gene(name):
    id_hits, matched_field = get_gene_ids(name)
    distinct_ids = []
    for hit in id_hits:
        if hit not in distinct_ids:
            distinct_ids.append(hit)

    return {
        "n_candidates": len(distinct_ids),
        "returned_id": distinct_ids[0] if len(distinct_ids) == 1 else "",
        "candidate_ids": ";".join(distinct_ids),
        "matched_field": matched_field,
    }

def handle_api_lookup(input, entity_type):
    result = None
    if entity_type == "chemical":
        result = resolve_chemical(input)
    elif entity_type == "gene":
        result = resolve_gene(input)
    else:
        raise ValueError(f"Invalid entity type: {entity_type}")
    return result

In [7]:
# Perform the API matching against the benchmark and create the table

def run_b3(data):
    api_lookup_result = []
    for row_id, name, entity_type in zip(data["id"], data["input"], data["entity_type"]):
        print(f"Looking up {row_id}")
        result = handle_api_lookup(name, entity_type)
        result["id"] = row_id
        result["input"] = name
        result["entity_type"] = entity_type

        api_lookup_result.append(result)
    return pd.DataFrame(api_lookup_result)

b3 = run_b3(benchmark)
b3.to_csv("../results/b3.csv", index=False)

Looking up CHEM-001
Looking up CHEM-002
Looking up CHEM-003
Looking up CHEM-004
Looking up CHEM-005
Looking up CHEM-006
Looking up CHEM-007
Looking up CHEM-008
Looking up CHEM-009
Looking up CHEM-010
Looking up CHEM-011
Looking up CHEM-012
Looking up CHEM-013
Looking up CHEM-014
Looking up CHEM-015
Looking up CHEM-016
Looking up CHEM-017
Looking up CHEM-018
Looking up CHEM-019
Looking up CHEM-020
Looking up CHEM-021
Looking up CHEM-022
Looking up CHEM-023
Looking up CHEM-024
Looking up CHEM-025
Looking up CHEM-026
Looking up CHEM-027
Looking up CHEM-028
Looking up CHEM-029
Looking up CHEM-030
Looking up CHEM-031
Looking up CHEM-032
Looking up CHEM-033
Looking up CHEM-034
Looking up CHEM-035
Looking up CHEM-036
Looking up CHEM-037
Looking up CHEM-038
Looking up CHEM-039
Looking up CHEM-040
Looking up CHEM-041
Looking up CHEM-042
Looking up CHEM-043
Looking up CHEM-044
Looking up CHEM-045
Looking up CHEM-046
Looking up CHEM-047
Looking up CHEM-048
Looking up CHEM-049
Looking up CHEM-050
